In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [10]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage

In [6]:
model = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=groq_api_key)

In [7]:
model.invoke([HumanMessage(content="Hi, My name is Prashant and I am Senior Software Engineer")])

AIMessage(content="Hello Prashant, nice to meet you. As a Senior Software Engineer, you must have a wealth of experience in designing, developing, and leading software projects. What technologies and domains do you specialize in? Are you currently working on any exciting projects that you'd like to talk about?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 49, 'total_tokens': 108, 'completion_time': 0.180647328, 'completion_tokens_details': None, 'prompt_time': 0.002235686, 'prompt_tokens_details': None, 'queue_time': 0.180737966, 'total_time': 0.182883014}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019db3f1-8de3-74c0-8962-6cb395bfc7b7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 59, 'total_tokens': 108})

In [11]:
model.invoke([
    HumanMessage(content="Hi, My name is Prashant and I am Senior Software Engineer"),
    AIMessage(content="Hello Prashant, nice to meet you. As a Senior Software Engineer, you must have a wealth of experience in designing, developing, and leading software projects. What technologies and domains do you specialize in? Are you currently working on any exciting projects that you'd like to talk about?"),
    HumanMessage(content="Hey, what is my name & what do I do?")
])

AIMessage(content='Your name is Prashant, and you are a Senior Software Engineer.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 129, 'total_tokens': 145, 'completion_time': 0.021428767, 'completion_tokens_details': None, 'prompt_time': 0.007045487, 'prompt_tokens_details': None, 'queue_time': 0.071931533, 'total_time': 0.028474254}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019db3f3-c51f-7153-a49b-b0b997f4e47d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 129, 'output_tokens': 16, 'total_tokens': 145})

## Message History

In [12]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [13]:
store = {}

In [14]:
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [15]:
config = {
    "configurable": {
        "session_id": "chat1"
    }
}

In [16]:
with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [18]:
response = with_message_history.invoke([
    HumanMessage(content="Hi, My name is Prashant and I am Senior Software Engineer")
], config=config)

In [19]:
response.content

"Hello Prashant, it's great to connect with you. As a Senior Software Engineer, you've likely worked on a wide range of projects and have a strong foundation in programming languages, software design patterns, and development methodologies. \n\nWhat specific areas of software engineering are you most passionate about, and what do you enjoy most about your work? Is it the problem-solving aspect, the collaborative team environment, or something else?"

In [20]:
with_message_history.invoke(
    [HumanMessage(content="What is my name & what do I do?")], config=config
)

AIMessage(content='Your name is Prashant, and you are a Senior Software Engineer.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 233, 'total_tokens': 249, 'completion_time': 0.018241431, 'completion_tokens_details': None, 'prompt_time': 0.022256962, 'prompt_tokens_details': None, 'queue_time': 0.054840017, 'total_time': 0.040498393}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019db3fb-2823-77e1-9a11-76f289572380-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 233, 'output_tokens': 16, 'total_tokens': 249})

## Prompt Templates

In [22]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer all the questions to the best of your ability"),
    MessagesPlaceholder(variable_name="messages")
])

chain = prompt | model

In [23]:
chain.invoke(
    {
        "messages": [HumanMessage(content="Hi, My name is Prashant")]
    }
)

AIMessage(content="Hello Prashant, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 59, 'total_tokens': 87, 'completion_time': 0.044274851, 'completion_tokens_details': None, 'prompt_time': 0.010951762, 'prompt_tokens_details': None, 'queue_time': 0.163630452, 'total_time': 0.055226613}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019db403-488f-7e43-93e8-a8474300d012-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens': 28, 'total_tokens': 87})

In [24]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [25]:
config = {
    "configurable": {
        "session_id": "chat3"
    }
}

response = with_message_history.invoke([HumanMessage(content="Hi, my name is Prashant")], config=config)

In [26]:
response.content

"Hello Prashant, it's nice to meet you. Is there something I can help you with or would you like to chat?"

#### Complex Example

In [27]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer all the questions to the best of your ability in {language}."),
    MessagesPlaceholder(variable_name="messages")
])

chain = prompt | model

In [28]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history, input_messages_key="messages")

In [29]:
config = {
    "configurable": {
        "session_id": "chat4"
    }
}

response = with_message_history.invoke({
    "messages" : [HumanMessage(content="Hi, my name is Prashant")],
    "language": "Hindi"
}, config=config)

response.content

'नमस्ते प्रशांत, मैं आपकी मदद करने के लिए तैयार हूँ। मैं आपके प्रश्नों का उत्तर देने की पूरी कोशिश करूँगा। आपको आज कैसे मदद कर सकता हूँ?'